# SMART — Multi-Object Tracker
**S**tick via **M**otion **A**nd **R**ecognition **T**racker

Bu notebook üç ana işlemi kapsar:
1. **Kurulum** — Repo, bağımlılıklar, DCNv2 derleme
2. **Eğitim** — MOT17 / SOMPT22 üzerinde `tracking,embedding` görevi
3. **İnference** — Video üzerinde çoklu nesne takibi

> **Gereksinim:** Çalışma zamanı → GPU (T4 veya daha iyisi)

## 0 · GPU & CUDA Kontrolü

In [ ]:
import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.version.cuda)
print('GPU     :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK — Runtime > GPU seç!')
assert torch.cuda.is_available(), 'GPU bulunamadı. Çalışma zamanını GPU olarak değiştir.'

## 1 · Google Drive Bağlantısı

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/SMART'   # ← İstediğin klasörü değiştir
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive klasörü:', DRIVE_ROOT)

## 2 · Repo Kurulumu

In [ ]:
%cd /content
!git clone https://github.com/sompt22/SMART.git
%cd /content/SMART

In [ ]:
# PyPI bağımlılıkları
!pip install -q \
    numpy==1.23.5 \
    opencv-python \
    Cython \
    numba \
    progress \
    matplotlib \
    easydict \
    scipy \
    pyyaml \
    yacs \
    cython-bbox \
    motmetrics \
    'lapx>=0.5.2' \
    openpyxl \
    Pillow \
    tensorboardX \
    fvcore \
    pycocotools \
    pyquaternion
print('Bağımlılıklar kuruldu.')

In [ ]:
# DCNv2 (Deformable Convolution) derle
# NOT: --user bayrağı Colab'da kurulum hatasını önler
%cd /content/SMART/src/lib/model/networks/DCNv2
!rm -rf build/ *.egg-info *.so
!python setup.py build develop --user 2>&1 | tail -10
print('DCNv2 derlendi.')

In [ ]:
# External NMS Cython modülünü derle
%cd /content/SMART/src/lib/external
!python setup.py build_ext --inplace 2>&1 | tail -5
print('External NMS derlendi.')

In [ ]:
# Python path'ini güncelle
import sys
for p in ['/content/SMART/src', '/content/SMART/src/lib']:
    if p not in sys.path:
        sys.path.insert(0, p)
print('sys.path güncellendi.')

## 3 · Veri Seti Hazırlama

**3A** → MOT17 (otomatik indir + COCO'ya dönüştür)  
**3B** → SOMPT22 / Özel Dataset (Drive'dan kopyala)

Birini seç ve çalıştır.

### 3A · MOT17

In [ ]:
import os

DATA_DIR  = '/content/SMART/data'
MOT17_DIR = f'{DATA_DIR}/mot17'
os.makedirs(f'{MOT17_DIR}/images', exist_ok=True)
os.makedirs(f'{MOT17_DIR}/annotations', exist_ok=True)

# Drive'da hazır zip varsa doğrudan kopyala, yoksa indir
MOT17_ZIP_DRIVE = f'{DRIVE_ROOT}/MOT17.zip'
TMP_ZIP = f'{MOT17_DIR}/MOT17.zip'

if os.path.exists(MOT17_ZIP_DRIVE):
    print('Drive\'dan kopyalanıyor...')
    !cp "{MOT17_ZIP_DRIVE}" "{TMP_ZIP}"
else:
    print('İndiriliyor (~4 GB)...')
    !wget -q --show-progress -O "{TMP_ZIP}" \
        https://motchallenge.net/data/MOT17.zip

!unzip -q "{TMP_ZIP}" -d "{MOT17_DIR}/images/"
# Zipten çıkan klasör adı 'MOT17' olabilir, düzelt
![ -d "{MOT17_DIR}/images/MOT17/train" ] && mv "{MOT17_DIR}/images/MOT17/train" "{MOT17_DIR}/images/train" || true
!rm -f "{TMP_ZIP}"
print('MOT17 açıldı.')
!ls "{MOT17_DIR}/images/"

In [ ]:
# MOT17 → COCO formatına dönüştür
# Script sabit yol kullanıyor, runtime'da patch ediyoruz
CONVERT_SCRIPT = '/content/SMART/src/tools/mot17/convert_mot_to_coco.py'

with open(CONVERT_SCRIPT) as f:
    code = f.read()

# Hardcoded mutlak yolları Colab yollarıyla değiştir
code = code.replace(
    "DATA_PATH = '/home/fatih/phd/SMART/data/mot17/images/'",
    f"DATA_PATH = '{MOT17_DIR}/images/'"
).replace(
    "OUT_PATH = DATA_PATH + 'annotations/'",
    f"OUT_PATH = '{MOT17_DIR}/annotations/'"
)

# __main__ guard Jupyter'da da geçer (kernel __name__ == '__main__')
exec(compile(code, CONVERT_SCRIPT, 'exec'))

print('\nAnnotation dosyaları:')
!ls -lh "{MOT17_DIR}/annotations/"

### 3B · SOMPT22 (Drive'dan)

In [ ]:
# Drive'daki beklenen yapı:
#   MyDrive/SMART/sompt22/
#     images/train/  ← sekans klasörleri
#     images/val/
#     annotations/train.json
#     annotations/val.json

SOMPT22_SRC = f'{DRIVE_ROOT}/sompt22'
SOMPT22_DST = '/content/SMART/data/sompt22'

if os.path.exists(SOMPT22_SRC):
    !cp -r "{SOMPT22_SRC}" "{SOMPT22_DST}"
    print('SOMPT22 kopyalandı.')
    !ls -lh "{SOMPT22_DST}/annotations/"
else:
    print(f'[UYARI] {SOMPT22_SRC} bulunamadı — Drive\'a yükle ve yolu güncelle.')

## 4 · Ön-eğitimli Model

In [ ]:
MODELS_DIR = '/content/SMART/models'
os.makedirs(MODELS_DIR, exist_ok=True)

PRETRAIN_DRIVE = f'{DRIVE_ROOT}/ctdet_coco_dla_2x.pth'
PRETRAIN_LOCAL = f'{MODELS_DIR}/ctdet_coco_dla_2x.pth'

if os.path.exists(PRETRAIN_DRIVE):
    !cp "{PRETRAIN_DRIVE}" "{PRETRAIN_LOCAL}"
    print('Model Drive\'dan kopyalandı.')
elif not os.path.exists(PRETRAIN_LOCAL):
    print('CenterNet DLA-34 COCO ağırlıkları indiriliyor...')
    !wget -q --show-progress -O "{PRETRAIN_LOCAL}" \
        https://dl.dropbox.com/s/shscf1k5qgkh3xm/ctdet_coco_dla_2x.pth
else:
    print('Model zaten mevcut.')

print('Model:', PRETRAIN_LOCAL)
!ls -lh "{MODELS_DIR}/"

## 5 · Eğitim Konfigürasyonu

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TÜM EĞİTİM PARAMETRELERİ — buradan düzenle
# ─────────────────────────────────────────────────────────────────────────────
import json

CFG = dict(
    task         = 'tracking,embedding',
    dataset      = 'mot17',          # 'mot17' | 'mot20' | 'sompt22' | 'divo'
    exp_id       = 'colab_mot17_dla34',

    # Model
    arch         = 'dla_34',
    load_model   = PRETRAIN_LOCAL,

    # Eğitim
    num_epochs   = 30,
    lr           = 1.25e-4,
    lr_step      = '20,25',
    save_point   = '20,25',
    batch_size   = 8,           # T4: 8, A100: 16
    num_workers  = 2,
    optim        = 'adam',      # 'adam' | 'adamw' | 'sgd'

    # Loss
    multi_loss       = 'uncertainty',
    embedding_loss   = 'focal',  # 'ce' | 'focal'
    embedding_weight = 0.1,
    know_dist_weight = 0.0,      # 0 = KD kapalı

    # Augmentation
    hm_disturb   = 0.05,
    lost_disturb = 0.4,
    fp_disturb   = 0.1,

    # Validasyon
    val_intervals = 5,

    # Freeze (tüm bileşenler serbest)
    # Seçici dondurma için True yap
    freeze_components = {
        'base':       False,
        'dla_up':     False,
        'ida_up':     False,
        'hm':         False,
        'reg':        False,
        'wh':         False,
        'ltrb_amodal': False,
        'embedding':  False,
        'tracking':   False,
    },

    gpus = '0',
)

# save_dir opts.py tarafından otomatik hesaplanır:
# /content/SMART/exp/<task>/<exp_id>/
SMART_EXP_DIR = f'/content/SMART/exp/{CFG["task"]}/{CFG["exp_id"]}'

print('Deney çıktı klasörü:', SMART_EXP_DIR)
for k, v in CFG.items():
    print(f'  {k:20s}: {v}')

## 6 · Eğitim

> **Not:** `opts.py`, çalışma dizinine (`cwd`) göre `root_dir`'i hesaplar.  
> Bu nedenle eğitim `/content/SMART/src/` içinden çalıştırılır.

In [ ]:
import sys, subprocess, os

# opts.py'nin root_dir'i doğru bulması için src/ içinden çalıştırılmalı
os.chdir('/content/SMART/src')

# freeze_components JSON string olarak opts'a geçirilir
freeze_json = json.dumps(CFG['freeze_components'])

# args listesi kullanıldığında shell quoting sorunu olmaz
train_args = [
    sys.executable, 'main.py', CFG['task'],
    '--exp_id',           CFG['exp_id'],
    '--dataset',          CFG['dataset'],
    '--arch',             CFG['arch'],
    '--load_model',       CFG['load_model'],
    '--num_epochs',       str(CFG['num_epochs']),
    '--lr',               str(CFG['lr']),
    '--lr_step',          CFG['lr_step'],
    '--save_point',       CFG['save_point'],
    '--batch_size',       str(CFG['batch_size']),
    '--num_workers',      str(CFG['num_workers']),
    '--optim',            CFG['optim'],
    '--multi_loss',       CFG['multi_loss'],
    '--embedding_loss',   CFG['embedding_loss'],
    '--embedding_weight', str(CFG['embedding_weight']),
    '--know_dist_weight', str(CFG['know_dist_weight']),
    '--hm_disturb',       str(CFG['hm_disturb']),
    '--lost_disturb',     str(CFG['lost_disturb']),
    '--fp_disturb',       str(CFG['fp_disturb']),
    '--val_intervals',    str(CFG['val_intervals']),
    '--gpus',             CFG['gpus'],
    '--num_classes',      '1',
    '--freeze_components', freeze_json,
    '--ltrb_amodal',
    '--pre_hm',
    '--same_aug_pre',     # NOT: train.sh'deki --same_aug yanlış; doğrusu bu
]

print('Komut:', ' '.join(train_args[:6]), '...')
proc = subprocess.Popen(train_args, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('\nEğitim tamamlandı. Return code:', proc.returncode)

In [ ]:
# Eğitimi kaldığı yerden devam ettir (--resume)
os.chdir('/content/SMART/src')

resume_args = train_args + ['--resume']
proc = subprocess.Popen(resume_args, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('\nResume tamamlandı. Return code:', proc.returncode)

## 7 · TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{SMART_EXP_DIR}"

## 8 · Modeli Drive'a Yedekle

In [ ]:
if os.path.exists(SMART_EXP_DIR):
    BACKUP_DIR = f'{DRIVE_ROOT}/checkpoints/{CFG["exp_id"]}'
    os.makedirs(BACKUP_DIR, exist_ok=True)
    !cp "{SMART_EXP_DIR}"/*.pth "{BACKUP_DIR}/" 2>/dev/null || echo 'pth yok'
    !cp "{SMART_EXP_DIR}"/*.log "{BACKUP_DIR}/" 2>/dev/null || echo 'log yok'
    print('Yedekleme tamamlandı:', BACKUP_DIR)
    !ls -lh "{BACKUP_DIR}/"
else:
    print('[UYARI] Deney klasörü yok:', SMART_EXP_DIR)

## 9 · İnference (Video Üzerinde MOT)

In [ ]:
# Model: eğitimden çıkan son model
INFER_MODEL = f'{SMART_EXP_DIR}/model_last.pth'

# Yoksa Drive'dan kullan:
# INFER_MODEL = f'{DRIVE_ROOT}/checkpoints/{CFG["exp_id"]}/model_last.pth'

assert os.path.exists(INFER_MODEL), f'Model bulunamadı: {INFER_MODEL}'
print('Model:', INFER_MODEL)

# Test videosu
VIDEO_SRC = f'{DRIVE_ROOT}/test_video.mp4'  # ← Drive'daki video
VIDEO_DST = '/content/test_video.mp4'

if os.path.exists(VIDEO_SRC):
    !cp "{VIDEO_SRC}" "{VIDEO_DST}"
    print('Video kopyalandı.')
else:
    # MOT17 sekansından kısa clip oluştur (Drive yoksa)
    SEQ_IMGS = f'{MOT17_DIR}/images/train/MOT17-02-FRCNN/img1'
    if os.path.exists(SEQ_IMGS):
        !ffmpeg -y -loglevel error -framerate 30 -pattern_type glob \
            -i "{SEQ_IMGS}/*.jpg" -vcodec libx264 -t 10 "{VIDEO_DST}"
        print('Örnek video oluşturuldu:', VIDEO_DST)
    else:
        print('[UYARI] Video bulunamadı. VIDEO_SRC veya MOT17 yolunu kontrol et.')

In [ ]:
# İnference
os.chdir('/content/SMART/src')

DEMO_EXP = 'colab_inference'

demo_args = [
    sys.executable, 'demo.py', CFG['task'],
    '--exp_id',      DEMO_EXP,
    '--load_model',  INFER_MODEL,
    '--demo',        VIDEO_DST,
    '--num_classes', '1',
    '--gpus',        '0',
    '--track_thresh', '0.4',
    '--pre_thresh',   '0.5',
    '--max_age',      '50',
    '--debug',        '0',
    '--ltrb_amodal',
    '--save_video',
    '--no_pause',
]

proc = subprocess.Popen(demo_args, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('\nİnference tamamlandı. Return code:', proc.returncode)

In [ ]:
# Çıktı videosunu Colab'da oynat ve Drive'a kaydet
import glob
from IPython.display import Video, display

DEMO_OUT = f'/content/SMART/exp/{CFG["task"]}/{DEMO_EXP}'
output_videos = glob.glob(f'{DEMO_OUT}/**/*.avi', recursive=True) + \
                glob.glob(f'{DEMO_OUT}/**/*.mp4', recursive=True)

if output_videos:
    OUT_VIDEO = output_videos[0]
    OUT_MP4 = OUT_VIDEO.rsplit('.', 1)[0] + '_colab.mp4'
    # Colab'ın H.264 oynatıcısı için dönüştür
    !ffmpeg -y -loglevel error -i "{OUT_VIDEO}" -vcodec libx264 "{OUT_MP4}"
    display(Video(OUT_MP4, embed=True, width=800))
    !cp "{OUT_MP4}" "{DRIVE_ROOT}/"
    print('Drive\'a kaydedildi:', DRIVE_ROOT)
else:
    print('[UYARI] Çıktı videosu bulunamadı. DEMO_OUT:', DEMO_OUT)
    !ls -lR "{DEMO_OUT}" 2>/dev/null || echo 'Klasör yok.'

## 10 · Değerlendirme (MOT Metrikleri)

In [ ]:
os.chdir('/content/SMART/src')

eval_args = [
    sys.executable, 'test.py', CFG['task'],
    '--exp_id',       'colab_eval',
    '--dataset',      CFG['dataset'],
    '--load_model',   INFER_MODEL,
    '--num_classes',  '1',
    '--gpus',         '0',
    '--track_thresh', '0.4',
    '--pre_thresh',   '0.5',
    '--max_age',      '150',
    '--ltrb_amodal',
    '--pre_hm',
    '--trainval',
]

proc = subprocess.Popen(eval_args, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('\nDeğerlendirme tamamlandı. Return code:', proc.returncode)

## 11 · Tek Kare Görselleştirme

In [ ]:
import cv2, glob, numpy as np
import matplotlib.pyplot as plt
import torch, sys, os

# opts.py root_dir'i cwd'den hesaplar; src/ içinden çalıştırılmalı
os.chdir('/content/SMART/src')

from opts import opts
from detector import Detector

opt = opts().parse([
    CFG['task'],
    '--load_model',  INFER_MODEL,
    '--dataset',     CFG['dataset'],
    '--num_classes', '1',
    '--gpus',        '0',
    '--track_thresh','0.4',
    '--pre_thresh',  '0.5',
    '--max_age',     '50',
    '--ltrb_amodal',
    '--no_pause',
    '--debug',       '0',
])

detector = Detector(opt)

# İlk MOT17 karesini yükle
img_files = sorted(glob.glob(f'{MOT17_DIR}/images/train/MOT17-02-FRCNN/img1/*.jpg'))
assert img_files, 'Görüntü bulunamadı — MOT17 yolunu kontrol et.'
img = cv2.imread(img_files[0])

ret = detector.run(img)
results = ret['results']
print(f'Tespit: {len(results)} nesne | Toplam süre: {ret["tot"]:.3f}s')

vis = img.copy()
for det in results:
    x1, y1, x2, y2 = [int(v) for v in det['bbox']]
    tid   = det.get('tracking_id', -1)
    score = det['score']
    color = tuple(int(c * 255) for c in plt.cm.hsv(((tid * 37) % 256) / 256)[:3])
    cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)
    cv2.putText(vis, f'ID:{tid} {score:.2f}', (x1, y1 - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

plt.figure(figsize=(16, 9))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title(f'{len(results)} detections on {os.path.basename(img_files[0])}')
plt.tight_layout()
plt.show()